In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
from pathlib import Path

from audio_classification_playground.synthetic.generator import generate_synthetic_audio
from audio_classification_playground.synthetic import visualize_synthetic

In [ ]:
res = generate_synthetic_audio(
    output_dir=Path("../outputs"),
    resources_dir=Path("../resources"),
)

In [ ]:
from audio_classification_playground.synthetic import visualize_synthetic


visualize_synthetic.plot_labels(
    labels=res.labels,
)

In [ ]:
visualize_synthetic.plot_labels_combined(
    labels=res.labels,
)

In [ ]:
from audio_classification_playground.panns import inference

In [ ]:
model = inference.load_model(checkpoint_path=Path("../resources/models/Cnn14_DecisionLevelMax_mAP=0.385.pth"))

In [ ]:
frame_wise_output = inference.frame_level_inference(model, res.audio)

In [ ]:
from audio_classification_playground.panns.visualize import visualize_frame_level_output,visualize_high_level_categories
visualize_frame_level_output(
    raw_audio=res.audio,
    framewise_output=frame_wise_output,
)

In [ ]:
visualize_high_level_categories(
    raw_audio=res.audio,
    framewise_output=frame_wise_output,
)

In [ ]:
import requests
ontology_url = "https://raw.githubusercontent.com/audioset/ontology/master/ontology.json"

ontology = requests.get(ontology_url).json()
id_to_name = {x['id']: x['name'] for x in ontology}

In [ ]:
# Build parent lookup
child_to_parents = {}
id_to_name = {}

for node in ontology:
    id_to_name[node['id']] = node['name']
    for child_id in node.get('child_ids', []):
        if child_id not in child_to_parents:
            child_to_parents[child_id] = []
        child_to_parents[child_id].append(node['id'])

# Function to get all ancestors
def get_ancestors(class_id):
    ancestors = set()
    queue = [class_id]
    while queue:
        current = queue.pop(0)
        if current in child_to_parents:
            for parent in child_to_parents[current]:
                if parent not in ancestors:
                    ancestors.add(parent)
                    queue.append(parent)
    return ancestors

# Map to Speech/Music/SFX
def map_to_high_level(class_id):
    ancestors = get_ancestors(class_id)
    ancestor_names = {id_to_name[aid] for aid in ancestors}
    
    if 'Music' in ancestor_names:
        return 'Music'
    elif 'Speech' in ancestor_names or 'Human voice' in ancestor_names:
        return 'Speech'
    else:
        return 'SFX'  # Default to sound effects

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from audio_classification_playground.panns.config import index_to_category

# ─── Config ──────────────────────────────────────────────────────────────────
PROB_THRESHOLD = 0.05
FRAMES_PER_SECOND = 100  # 32000 // 320

framewise_output = frame_wise_output  # (num_frames, 527)
num_frames = framewise_output.shape[0]
total_duration_sec = num_frames / FRAMES_PER_SECOND
time_axis = np.arange(num_frames) / FRAMES_PER_SECOND

# ─── Requirement 1 — Ground truth regions (fill in your values) ─────────────
ground_truth = {
    "Music": [[0.0, 5.0], [12.0, 18.0]],   # ← replace with your actual regions
    "SFX":   [[6.0, 11.0], [19.0, 22.0]],   # ← replace with your actual regions
}
print(f"Total audio duration: {total_duration_sec:.2f}s")
for cat, regions in ground_truth.items():
    print(f"  {cat}: {regions}")

# ─── Requirement 2 — Aggregated signals with threshold ──────────────────────
# Collect class indices per category
category_indices = {"Music": [], "SFX": []}
for class_idx, cat in enumerate(index_to_category):
    if cat in category_indices:
        category_indices[cat].append(class_idx)

signals = {}
for cat, indices in category_indices.items():
    indices = np.array(indices)
    probs = framewise_output[:, indices].copy()        # (num_frames, n_classes)
    probs[probs < PROB_THRESHOLD] = 0.0                # zero out below threshold
    signals[cat] = probs.sum(axis=1)                   # (num_frames,)

# Plot
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
for ax, cat in zip(axes, ["Music", "SFX"]):
    ax.plot(time_axis, signals[cat], linewidth=0.8)
    ax.set_ylabel(f"{cat} signal")
    ax.set_title(f"{cat} — aggregated (threshold={PROB_THRESHOLD})")
    # shade ground truth regions
    for start, end in ground_truth[cat]:
        ax.axvspan(start, end, alpha=0.15, color="green", label="GT active")
    ax.legend(loc="upper right")
axes[-1].set_xlabel("Time (seconds)")
plt.tight_layout()
plt.show()

# ─── Requirement 3 — Statistics per ground truth region ──────────────────────
def _build_mask(regions: list[list[float]], num_frames: int, fps: int) -> np.ndarray:
    """Boolean mask: True for frames inside any of the given time regions."""
    mask = np.zeros(num_frames, dtype=bool)
    for start, end in regions:
        mask[int(start * fps): int(end * fps)] = True
    return mask

for cat in ["Music", "SFX"]:
    sig = signals[cat]
    active_mask = _build_mask(ground_truth[cat], num_frames, FRAMES_PER_SECOND)
    inactive_mask = ~active_mask

    active_vals = sig[active_mask]
    inactive_vals = sig[inactive_mask]

    print(f"\n{'='*50}")
    print(f"  {cat}")
    print(f"{'='*50}")
    if active_vals.size:
        print(f"  Active   — median: {np.median(active_vals):.4f}, "
              f"p10: {np.percentile(active_vals, 10):.4f}, "
              f"p90: {np.percentile(active_vals, 90):.4f}")
    else:
        print("  Active   — no frames")
    if inactive_vals.size:
        print(f"  Inactive — median: {np.median(inactive_vals):.4f}, "
              f"p90: {np.percentile(inactive_vals, 90):.4f}, "
              f"max: {np.max(inactive_vals):.4f}")
    else:
        print("  Inactive — no frames")